In [4]:
import os
import whisper
import pandas as pd
import re


# Load the Whisper model
model = whisper.load_model("large")  # Using the "large" model

# Set file paths
FILE_PATH = r"c:\Year 2\Y2- Block C\The Blind Date Show 2 - Episode 20 with Malak & Sabbagh.mp3"
OUTPUT_CSV = r"transcribed_data_whisper.csv"

# Transcription function using Whisper
def transcribe_with_whisper(file_path):
    result = model.transcribe(file_path, language="ar")
    return result['text']

# Text cleaning functions
def split_text_into_sentences(text, max_word_count=20):
    sentences = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            parts = re.split(r'(?<=[\.!\؟؛])\s+', line)
            sentences.extend([part.strip() for part in parts if part.strip()])
    
    refined_sentences = []
    for sentence in sentences:
        words = sentence.split()
        if len(words) > max_word_count:
            subparts = re.split(r'(?<=[,،])\s+', sentence)
            for sp in subparts:
                sp = sp.strip()
                if sp:
                    sub_words = sp.split()
                    if len(sub_words) > max_word_count:
                        for i in range(0, len(sub_words), max_word_count):
                            refined_sentences.append(" ".join(sub_words[i:i+max_word_count]))
                    else:
                        refined_sentences.append(sp)
        else:
            refined_sentences.append(sentence)
    return refined_sentences

def merge_short_fragments(sentences, min_word_count=3):
    merge_starters = {'و', 'زواج', 'لكن', 'بس', 'كبير', 'كتير'}
    merged = []
    for s in sentences:
        s = s.strip()
        if merged:
            words = s.split()
            if len(words) < min_word_count or (words and words[0] in merge_starters):
                merged[-1] = merged[-1] + " " + s
                continue
        merged.append(s)
    return merged

def remove_english_artifacts(text):
    cleaned = re.sub(r'\b[A-Za-z]+\b', '', text)
    cleaned = re.sub(r'\s{2,}', ' ', cleaned)
    return cleaned.strip()

def clean_sentence(sentence):
    sentence = re.sub(r'^[,،\s]+', '', sentence)
    sentence = re.sub(r'[,،\s]+$', '', sentence)
    return sentence.strip()

def save_sentences_to_csv(sentences, file_path):
    df = pd.DataFrame(sentences, columns=["Sentence"])
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"✅ Transcribed text saved to: {file_path}")

# Run the pipeline
def run_transcription():
    if not os.path.isfile(FILE_PATH):
        print("❌ File not found:", FILE_PATH)
        return

    print("🎙 Transcribing audio with Whisper...")
    transcript_text = transcribe_with_whisper(FILE_PATH)

    print("🔍 Removing English artifacts...")
    cleaned_text = remove_english_artifacts(transcript_text)

    print("📌 Splitting into sentences...")
    sentences = split_text_into_sentences(cleaned_text)
    sentences = merge_short_fragments(sentences)

    sentences = [clean_sentence(s) for s in sentences if s]

    print("💾 Saving to CSV...")
    save_sentences_to_csv(sentences, OUTPUT_CSV)

# Run the function
run_transcription()

100%|█████████████████████████████████████| 2.88G/2.88G [02:08<00:00, 24.0MiB/s]


🎙 Transcribing audio with Whisper...
🔍 Removing English artifacts...
📌 Splitting into sentences...
💾 Saving to CSV...
✅ Transcribed text saved to: transcribed_data_whisper.csv
